In [ ]:
from google.colab import drive
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

drive.mount("/content/drive")

spark = (
    SparkSession.builder
    .appName("CloudProviderAnalytics_Gold_FinOps")
    .master("local[*]")
    .getOrCreate()
)

DATA_PATH = Path(
    "/content/drive/MyDrive/cloud-provider-analytics-data"
)

SILVER_PATH = DATA_PATH / "silver"
GOLD_PATH = DATA_PATH / "gold"

print("Spark:", spark.version)
print("Existe Silver:", SILVER_PATH.exists())


Mounted at /content/drive
Spark: 4.0.5
Existe Silver: True


In [ ]:
events_silver = spark.read.parquet(
    str(SILVER_PATH / "usage_events")
)

print("Total de eventos:", events_silver.count())

events_silver.select(
    "event_id",
    "timestamp",
    "org_id",
    "service",
    "cost_usd_increment",
    "negative_cost_flag"
).show(5, truncate=False)


Total de eventos: 43200
+----------------+-------------------+------------+----------+------------------+------------------+
|event_id        |timestamp          |org_id      |service   |cost_usd_increment|negative_cost_flag|
+----------------+-------------------+------------+----------+------------------+------------------+
|evt_5kgf39wf58yj|2025-08-31 03:42:00|org_pja1wj0t|networking|1.1427            |false             |
|evt_o4m8mz9v6zsm|2025-08-21 01:53:00|org_tvhhpbmy|networking|1.273             |false             |
|evt_gapdt3d4bwip|2025-07-15 01:18:00|org_5935a0l7|database  |4.406             |false             |
|evt_m2pae4pmrqiw|2025-07-14 16:19:00|org_53lc58dr|compute   |0.0989            |false             |
|evt_vzg86dq8g2k1|2025-08-30 17:05:00|org_h2m5zja8|networking|0.0655            |false             |
+----------------+-------------------+------------+----------+------------------+------------------+
only showing top 5 rows


In [ ]:
events_finops = events_silver.withColumn(
    "event_date",
    F.to_date("timestamp")
)

print(
    "Eventos sin fecha:",
    events_finops.filter(F.col("event_date").isNull()).count()
)

print(
    "Eventos sin costo:",
    events_finops.filter(
        F.col("cost_usd_increment").isNull()
    ).count()
)

print(
    "Eventos con costo negativo:",
    events_finops.filter(
        F.col("negative_cost_flag")
    ).count()
)

print("\nRango de fechas:")
events_finops.select(
    F.min("event_date").alias("primera_fecha"),
    F.max("event_date").alias("ultima_fecha")
).show()


Eventos sin fecha: 0
Eventos sin costo: 0
Eventos con costo negativo: 216

Rango de fechas:
+-------------+------------+
|primera_fecha|ultima_fecha|
+-------------+------------+
|   2025-07-03|  2025-08-31|
+-------------+------------+



In [ ]:
# Crear mart FinOps diario por organización y servicio

gold_finops = (
    events_finops
    .groupBy("org_id", "event_date", "service")
    .agg(
        F.count("*").alias("total_events"),
        F.round(F.sum("cost_usd_increment"), 4).alias("total_cost_usd"),
        F.sum("requests").alias("requests"),
        F.sum("carbon_kg").alias("carbon_kg"),
        F.count_if("negative_cost_flag").alias("negative_cost_events"),
        F.count_if("missing_usage_data").alias("incomplete_usage_events"),
    )
)

gold_finops.agg(
    F.count("*").alias("filas_gold"),
    F.sum("total_events").alias("eventos_representados"),
    F.round(F.sum("total_cost_usd"), 4).alias("costo_total_usd"),
    F.round(F.sum("requests"), 4).alias("requests_total"),
    F.round(F.sum("carbon_kg"), 4).alias("carbon_kg_total"),
).show()


+----------+---------------------+---------------+--------------+---------------+
|filas_gold|eventos_representados|costo_total_usd|requests_total|carbon_kg_total|
+----------+---------------------+---------------+--------------+---------------+
|     11050|                43200|    147433.9778|     2186404.0|       367.5024|
+----------+---------------------+---------------+--------------+---------------+



In [ ]:
# Comparar cantidad de eventos

total_silver = events_finops.count()

total_gold = gold_finops.agg(
    F.sum("total_events")
).first()[0]

# Comparar costos

costo_silver = events_finops.agg(
    F.sum("cost_usd_increment")
).first()[0]

costo_gold = gold_finops.agg(
    F.sum("total_cost_usd")
).first()[0]

print("Eventos Silver:", total_silver)
print("Eventos representados en Gold:", total_gold)

print("\nCosto total Silver:", costo_silver)
print("Costo total Gold:", costo_gold)

print(
    "\nDiferencia de costos:",
    round(costo_silver - costo_gold, 4)
)


Eventos Silver: 43200
Eventos representados en Gold: 43200

Costo total Silver: 147433.97780000017
Costo total Gold: 147433.97779999883

Diferencia de costos: 0.0


In [ ]:
# Ruta de salida del mart FinOps

ruta_gold_finops = GOLD_PATH / "org_daily_usage_by_service"

# Guardar los resultados en formato Parquet

(
    gold_finops
    .write
    .mode("overwrite")
    .parquet(str(ruta_gold_finops))
)

print("Mart FinOps guardado correctamente en:")
print(ruta_gold_finops)


Mart FinOps guardado correctamente en:
/content/drive/MyDrive/cloud-provider-analytics-data/gold/org_daily_usage_by_service


In [ ]:
# Volver a leer el mart desde Google Drive

gold_finops_check = spark.read.parquet(str(ruta_gold_finops))

gold_finops_check.agg(
    F.count("*").alias("filas_gold"),
    F.sum("total_events").alias("eventos_representados"),
    F.round(F.sum("total_cost_usd"), 4).alias("costo_total_usd"),
    F.round(F.sum("requests"), 4).alias("requests_total"),
    F.round(F.sum("carbon_kg"), 4).alias("carbon_kg_total"),
).show()


+----------+---------------------+---------------+--------------+---------------+
|filas_gold|eventos_representados|costo_total_usd|requests_total|carbon_kg_total|
+----------+---------------------+---------------+--------------+---------------+
|     11050|                43200|    147433.9778|     2186404.0|       367.5024|
+----------+---------------------+---------------+--------------+---------------+



In [ ]:
# Analizar costos por servicio desde Gold

costos_por_servicio = (
    gold_finops_check
    .groupBy("service")
    .agg(
        F.round(
            F.sum("total_cost_usd"), 2
        ).alias("costo_total_usd"),

        F.sum("total_events").alias("total_eventos"),

        F.sum(
            "negative_cost_events"
        ).alias("eventos_costo_negativo"),

        F.sum(
            "incomplete_usage_events"
        ).alias("eventos_uso_incompleto")
    )
    .orderBy(F.desc("costo_total_usd"))
)

costos_por_servicio.show(truncate=False)


+----------+---------------+-------------+----------------------+----------------------+
|service   |costo_total_usd|total_eventos|eventos_costo_negativo|eventos_uso_incompleto|
+----------+---------------+-------------+----------------------+----------------------+
|compute   |61786.8        |12498        |56                    |1237                  |
|genai     |29544.43       |4158         |21                    |410                   |
|database  |22635.46       |7419         |44                    |740                   |
|analytics |19708.76       |4590         |19                    |411                   |
|storage   |9407.14        |7614         |44                    |726                   |
|networking|4351.39        |6921         |32                    |640                   |
+----------+---------------+-------------+----------------------+----------------------+



In [ ]:
# Ranking de organizaciones por costo total

costos_por_organizacion = (
    gold_finops_check
    .groupBy("org_id")
    .agg(
        F.round(
            F.sum("total_cost_usd"), 2
        ).alias("costo_total_usd"),

        F.sum("total_events").alias("total_eventos"),

        F.sum(
            "negative_cost_events"
        ).alias("eventos_costo_negativo")
    )
    .orderBy(F.desc("costo_total_usd"))
)

print(
    "Organizaciones con eventos:",
    costos_por_organizacion.count()
)

print("\nTop 10 organizaciones por costo:")
costos_por_organizacion.show(10, truncate=False)


Organizaciones con eventos: 80

Top 10 organizaciones por costo:
+------------+---------------+-------------+----------------------+
|org_id      |costo_total_usd|total_eventos|eventos_costo_negativo|
+------------+---------------+-------------+----------------------+
|org_53lc58dr|4380.92        |1113         |4                     |
|org_chj755nf|4342.23        |963          |8                     |
|org_kdgigatj|4321.53        |1227         |5                     |
|org_pbhsahxt|4077.16        |1098         |8                     |
|org_ktakpuxq|3546.84        |858          |3                     |
|org_rixa11dp|3136.49        |567          |3                     |
|org_sg65kxvf|3083.96        |756          |10                    |
|org_c11ertj5|2940.26        |642          |6                     |
|org_dppq0y9d|2887.66        |540          |3                     |
|org_5935a0l7|2787.43        |729          |4                     |
+------------+---------------+-------------+-------

In [ ]:
# Leer organizaciones desde Bronze

BRONZE_PATH = DATA_PATH / "bronze"

customers_bronze = spark.read.parquet(
    str(BRONZE_PATH / "customers_orgs")
)

print("Organizaciones en Bronze:", customers_bronze.count())

print("Columnas disponibles:")
print(customers_bronze.columns)

customers_bronze.show(5, truncate=False)


Organizaciones en Bronze: 80
Columnas disponibles:
['org_id', 'org_name', 'industry', 'hq_region', 'plan_tier', 'is_enterprise', 'signup_date', 'sales_rep', 'lifecycle_stage', 'marketing_source', 'nps_score', 'ingest_ts', 'source_file']
+------------+---------------+-------------+----------+---------+-------------+-----------+---------+---------------+----------------+---------+--------------------------+--------------------------------------------------------------------------------------+
|org_id      |org_name       |industry     |hq_region |plan_tier|is_enterprise|signup_date|sales_rep|lifecycle_stage|marketing_source|nps_score|ingest_ts                 |source_file                                                                           |
+------------+---------------+-------------+----------+---------+-------------+-----------+---------+---------------+----------------+---------+--------------------------+--------------------------------------------------------------------------

In [ ]:
# Seleccionar información de las organizaciones

customers_info = customers_bronze.select(
    "org_id",
    "org_name",
    "industry",
    "hq_region",
    "plan_tier"
)

# Relacionar costos con organizaciones

costos_org_detalle = (
    costos_por_organizacion
    .join(
        customers_info,
        on="org_id",
        how="left"
    )
)

# Mostrar ranking con nombres

costos_org_detalle.select(
    "org_name",
    "industry",
    "plan_tier",
    "costo_total_usd",
    "total_eventos"
).orderBy(
    F.desc("costo_total_usd")
).show(10, truncate=False)


+----------------+-------------+---------+---------------+-------------+
|org_name        |industry     |plan_tier|costo_total_usd|total_eventos|
+----------------+-------------+---------+---------------+-------------+
|Apex Digital 16 |Energy       |standard |4380.92        |1113         |
|Delta Tech 32   |Fintech      |pro      |4342.23        |963          |
|Zenith Tech 79  |Gaming       |standard |4321.53        |1227         |
|Nova Tech 1     |Education    |standard |4077.16        |1098         |
|Delta Digital 49|Fintech      |pro      |3546.84        |858          |
|Vertex AI 53    |Government   |standard |3136.49        |567          |
|Omega Data 14   |E-commerce   |standard |3083.96        |756          |
|Apex Cloud 17   |Retail       |pro      |2940.26        |642          |
|Vertex Labs 22  |Manufacturing|standard |2887.66        |540          |
|Omega Digital 40|Gaming       |standard |2787.43        |729          |
+----------------+-------------+---------+---------

In [ ]:
print(
    "Organizaciones originales:",
    costos_por_organizacion.count()
)

print(
    "Organizaciones después del join:",
    costos_org_detalle.count()
)

print(
    "Organizaciones sin nombre:",
    costos_org_detalle.filter(
        F.col("org_name").isNull()
    ).count()
)

print(
    "Costo total después del join:",
    costos_org_detalle.agg(
        F.round(
            F.sum("costo_total_usd"), 2
        )
    ).first()[0]
)


Organizaciones originales: 80
Organizaciones después del join: 80
Organizaciones sin nombre: 0
Costo total después del join: 147434.02


In [ ]:
# Crear mart de costos por organización
# sin redondear previamente a dos decimales

gold_org_costs = (
    gold_finops_check
    .groupBy("org_id")
    .agg(
        F.round(
            F.sum("total_cost_usd"), 4
        ).alias("total_cost_usd"),

        F.sum("total_events").alias("total_events"),

        F.sum(
            "negative_cost_events"
        ).alias("negative_cost_events"),

        F.sum(
            "incomplete_usage_events"
        ).alias("incomplete_usage_events")
    )
    .join(
        customers_info,
        on="org_id",
        how="left"
    )
)

print("Organizaciones:", gold_org_costs.count())

print(
    "Costo total USD:",
    gold_org_costs.agg(
        F.sum("total_cost_usd")
    ).first()[0]
)

print(
    "Eventos representados:",
    gold_org_costs.agg(
        F.sum("total_events")
    ).first()[0]
)

print(
    "Organizaciones sin nombre:",
    gold_org_costs.filter(
        F.col("org_name").isNull()
    ).count()
)

gold_org_costs.select(
    "org_name",
    "industry",
    "plan_tier",
    "total_cost_usd",
    "total_events"
).orderBy(
    F.desc("total_cost_usd")
).show(10, truncate=False)


Organizaciones: 80
Costo total USD: 147433.9778
Eventos representados: 43200
Organizaciones sin nombre: 0
+----------------+-------------+---------+--------------+------------+
|org_name        |industry     |plan_tier|total_cost_usd|total_events|
+----------------+-------------+---------+--------------+------------+
|Apex Digital 16 |Energy       |standard |4380.9198     |1113        |
|Delta Tech 32   |Fintech      |pro      |4342.2291     |963         |
|Zenith Tech 79  |Gaming       |standard |4321.5326     |1227        |
|Nova Tech 1     |Education    |standard |4077.1617     |1098        |
|Delta Digital 49|Fintech      |pro      |3546.8375     |858         |
|Vertex AI 53    |Government   |standard |3136.4901     |567         |
|Omega Data 14   |E-commerce   |standard |3083.9642     |756         |
|Apex Cloud 17   |Retail       |pro      |2940.2553     |642         |
|Vertex Labs 22  |Manufacturing|standard |2887.6574     |540         |
|Omega Digital 40|Gaming       |standard |

In [ ]:
# Definir ruta del segundo mart Gold

ruta_gold_org = GOLD_PATH / "org_cost_summary"

# Guardar en formato Parquet

(
    gold_org_costs
    .write
    .mode("overwrite")
    .parquet(str(ruta_gold_org))
)

print("Mart guardado correctamente en:")
print(ruta_gold_org)


Mart guardado correctamente en:
/content/drive/MyDrive/cloud-provider-analytics-data/gold/org_cost_summary


In [ ]:
# Volver a leer desde Google Drive

gold_org_check = spark.read.parquet(
    str(ruta_gold_org)
)

print("Organizaciones:", gold_org_check.count())

print(
    "Eventos representados:",
    gold_org_check.agg(
        F.sum("total_events")
    ).first()[0]
)

print(
    "Costo total USD:",
    gold_org_check.agg(
        F.round(F.sum("total_cost_usd"), 4)
    ).first()[0]
)

print(
    "Eventos con costo negativo:",
    gold_org_check.agg(
        F.sum("negative_cost_events")
    ).first()[0]
)

print(
    "Eventos con uso incompleto:",
    gold_org_check.agg(
        F.sum("incomplete_usage_events")
    ).first()[0]
)


Organizaciones: 80
Eventos representados: 43200
Costo total USD: 147433.9778
Eventos con costo negativo: 216
Eventos con uso incompleto: 4164


In [ ]:
# Analizar costos por industria

costos_industria = (
    gold_org_check
    .groupBy("industry")
    .agg(
        F.countDistinct("org_id").alias(
            "total_organizaciones"
        ),

        F.round(
            F.sum("total_cost_usd"), 2
        ).alias("costo_total_usd"),

        F.round(
            F.avg("total_cost_usd"), 2
        ).alias("costo_promedio_org_usd"),

        F.sum("total_events").alias(
            "total_eventos"
        )
    )
    .orderBy(F.desc("costo_total_usd"))
)

costos_industria.show(truncate=False)


+-------------+--------------------+---------------+----------------------+-------------+
|industry     |total_organizaciones|costo_total_usd|costo_promedio_org_usd|total_eventos|
+-------------+--------------------+---------------+----------------------+-------------+
|Gaming       |12                  |24862.94       |2071.91               |7245         |
|Education    |10                  |17558.58       |1755.86               |5388         |
|Manufacturing|10                  |16612.22       |1661.22               |5109         |
|Government   |9                   |16188.69       |1798.74               |4857         |
|Retail       |9                   |15819.22       |1757.69               |4611         |
|Healthcare   |8                   |14127.71       |1765.96               |4077         |
|Fintech      |7                   |13598.33       |1942.62               |4110         |
|Energy       |5                   |11806.3        |2361.26               |3102         |
|E-commerc

In [ ]:
# Analizar costos por tipo de plan

costos_plan = (
    gold_org_check
    .groupBy("plan_tier")
    .agg(
        F.countDistinct("org_id").alias(
            "total_organizaciones"
        ),

        F.round(
            F.sum("total_cost_usd"), 2
        ).alias("costo_total_usd"),

        F.round(
            F.avg("total_cost_usd"), 2
        ).alias("costo_promedio_org_usd"),

        F.sum("total_events").alias(
            "total_eventos"
        )
    )
    .orderBy(F.desc("costo_total_usd"))
)

costos_plan.show(truncate=False)


+----------+--------------------+---------------+----------------------+-------------+
|plan_tier |total_organizaciones|costo_total_usd|costo_promedio_org_usd|total_eventos|
+----------+--------------------+---------------+----------------------+-------------+
|standard  |41                  |79398.08       |1936.54               |22836        |
|pro       |21                  |37156.68       |1769.37               |10983        |
|enterprise|10                  |18286.61       |1828.66               |5118         |
|free      |8                   |12592.61       |1574.08               |4263         |
+----------+--------------------+---------------+----------------------+-------------+



In [ ]:
# Leer facturación desde Silver

billing_silver = spark.read.parquet(
    str(SILVER_PATH / "billing_monthly")
)

print("Total de facturas:", billing_silver.count())

print("\nColumnas disponibles:")
billing_silver.printSchema()

print("\nPeríodos de facturación:")
billing_silver.groupBy("month").count().orderBy(
    "month"
).show(truncate=False)

print("\nMonedas utilizadas:")
billing_silver.groupBy("currency").count().show()

print("\nEjemplo de registros:")
billing_silver.show(5, truncate=False)


Total de facturas: 240

Columnas disponibles:
root
 |-- invoice_id: string (nullable = true)
 |-- org_id: string (nullable = true)
 |-- month: date (nullable = true)
 |-- subtotal: decimal(18,2) (nullable = true)
 |-- credits: decimal(18,2) (nullable = true)
 |-- taxes: decimal(18,2) (nullable = true)
 |-- currency: string (nullable = true)
 |-- exchange_rate_to_usd: decimal(18,8) (nullable = true)
 |-- ingest_ts: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- requires_review: boolean (nullable = true)
 |-- silver_processed_ts: timestamp (nullable = true)


Períodos de facturación:
+----------+-----+
|month     |count|
+----------+-----+
|2025-06-01|80   |
|2025-07-01|80   |
|2025-08-01|80   |
+----------+-----+


Monedas utilizadas:
+--------+-----+
|currency|count|
+--------+-----+
|     ARS|   51|
|     EUR|   29|
|     USD|  160|
+--------+-----+


Ejemplo de registros:
+------------+------------+----------+--------+-------+------+--------+-------------

In [ ]:
# Validaciones financieras de billing_monthly

print("Créditos nulos:",
    billing_silver.filter(
        F.col("credits").isNull()
    ).count()
)

print("Créditos positivos:",
    billing_silver.filter(
        F.col("credits") > 0
    ).count()
)

print("Créditos negativos:",
    billing_silver.filter(
        F.col("credits") < 0
    ).count()
)

print("Impuestos nulos:",
    billing_silver.filter(
        F.col("taxes").isNull()
    ).count()
)

print("Tipos de cambio nulos:",
    billing_silver.filter(
        F.col("exchange_rate_to_usd").isNull()
    ).count()
)

print("Tipos de cambio no positivos:",
    billing_silver.filter(
        F.col("exchange_rate_to_usd") <= 0
    ).count()
)

print("\nRangos de créditos por moneda:")

billing_silver.groupBy("currency").agg(
    F.min("credits").alias("credito_minimo"),
    F.max("credits").alias("credito_maximo"),
    F.count("credits").alias("creditos_informados")
).show(truncate=False)

print("\nRangos de tipos de cambio:")

billing_silver.groupBy("currency").agg(
    F.min("exchange_rate_to_usd").alias("tc_minimo"),
    F.max("exchange_rate_to_usd").alias("tc_maximo")
).show(truncate=False)


Créditos nulos: 137
Créditos positivos: 102
Créditos negativos: 0
Impuestos nulos: 0
Tipos de cambio nulos: 0
Tipos de cambio no positivos: 0

Rangos de créditos por moneda:
+--------+--------------+--------------+-------------------+
|currency|credito_minimo|credito_maximo|creditos_informados|
+--------+--------------+--------------+-------------------+
|ARS     |0.62          |55.70         |17                 |
|EUR     |0.62          |66.45         |15                 |
|USD     |0.00          |79.06         |71                 |
+--------+--------------+--------------+-------------------+


Rangos de tipos de cambio:
+--------+----------+----------+
|currency|tc_minimo |tc_maximo |
+--------+----------+----------+
|ARS     |0.00133000|0.00162000|
|EUR     |0.99810000|1.19808000|
|USD     |0.85463000|1.11791000|
+--------+----------+----------+



In [ ]:
# Revisar ejemplos de facturación por moneda

billing_silver.select(
    "invoice_id",
    "month",
    "currency",
    "subtotal",
    "credits",
    "taxes",
    "exchange_rate_to_usd",
    "requires_review"
).orderBy(
    "currency",
    "month"
).show(20, truncate=False)

print("\nFacturas USD con tipo de cambio distinto de 1:")

billing_silver.filter(
    (F.col("currency") == "USD")
    & (F.col("exchange_rate_to_usd") != 1)
).select(
    "invoice_id",
    "month",
    "subtotal",
    "exchange_rate_to_usd"
).show(10, truncate=False)

print(
    "Cantidad de facturas USD con TC distinto de 1:",
    billing_silver.filter(
        (F.col("currency") == "USD")
        & (F.col("exchange_rate_to_usd") != 1)
    ).count()
)


+------------+----------+--------+--------+-------+------+--------------------+---------------+
|invoice_id  |month     |currency|subtotal|credits|taxes |exchange_rate_to_usd|requires_review|
+------------+----------+--------+--------+-------+------+--------------------+---------------+
|inv_27nls4sz|2025-06-01|ARS     |520.91  |NULL   |109.39|0.00147000          |false          |
|inv_03ivwack|2025-06-01|ARS     |350.32  |NULL   |73.57 |0.00158000          |false          |
|inv_j2czafr0|2025-06-01|ARS     |816.10  |18.49  |171.38|0.00133000          |false          |
|inv_6m04yprp|2025-06-01|ARS     |465.63  |10.21  |97.78 |0.00147000          |false          |
|inv_mh4qx495|2025-06-01|ARS     |448.43  |NULL   |94.17 |0.00148000          |false          |
|inv_x23cus50|2025-06-01|ARS     |1299.22 |18.30  |272.84|0.00153000          |false          |
|inv_xltnvv1g|2025-06-01|ARS     |1238.82 |NULL   |260.15|0.00162000          |false          |
|inv_f4wiq8hl|2025-06-01|ARS     |862.20

In [ ]:
from pyspark.sql import functions as F

# Preparar importes para el análisis financiero

billing_prepared = (
    billing_silver

    # Identificar tipos de cambio USD inconsistentes
    .withColumn(
        "usd_exchange_rate_flag",
        (F.col("currency") == "USD")
        & (F.col("exchange_rate_to_usd") != 1)
    )

    # Definir tipo de cambio utilizado
    .withColumn(
        "applied_exchange_rate",
        F.when(
            F.col("currency") == "USD",
            F.lit(1)
        ).otherwise(
            F.col("exchange_rate_to_usd")
        )
    )

    # Registrar si el crédito está informado
    .withColumn(
        "credits_missing_flag",
        F.col("credits").isNull()
    )

    # Calcular subtotal en USD
    .withColumn(
        "subtotal_usd",
        F.round(
            F.col("subtotal")
            * F.col("applied_exchange_rate"),
            4
        )
    )

    # Calcular impuestos en USD
    .withColumn(
        "taxes_usd",
        F.round(
            F.col("taxes")
            * F.col("applied_exchange_rate"),
            4
        )
    )

    # Convertir créditos solo cuando están informados
    .withColumn(
        "credits_usd",
        F.when(
            F.col("credits").isNotNull(),
            F.round(
                F.col("credits")
                * F.col("applied_exchange_rate"),
                4
            )
        )
    )
)

print("Facturas preparadas:", billing_prepared.count())

print(
    "TC USD inconsistentes:",
    billing_prepared.filter(
        F.col("usd_exchange_rate_flag")
    ).count()
)

print(
    "Créditos faltantes:",
    billing_prepared.filter(
        F.col("credits_missing_flag")
    ).count()
)

billing_prepared.select(
    "invoice_id",
    "currency",
    "subtotal_usd",
    "credits_usd",
    "taxes_usd",
    "applied_exchange_rate"
).show(10, truncate=False)


Facturas preparadas: 240
TC USD inconsistentes: 160
Créditos faltantes: 137
+------------+--------+------------+-----------+---------+---------------------+
|invoice_id  |currency|subtotal_usd|credits_usd|taxes_usd|applied_exchange_rate|
+------------+--------+------------+-----------+---------+---------------------+
|inv_nym31sk0|USD     |1133.3400   |NULL       |238.0000 |1.00000000           |
|inv_540sejok|USD     |619.1700    |NULL       |130.0300 |1.00000000           |
|inv_ilp4vn2v|ARS     |0.8653      |NULL       |0.1817   |0.00160000           |
|inv_27nls4sz|ARS     |0.7657      |NULL       |0.1608   |0.00147000           |
|inv_xq1k1qrr|EUR     |999.8702    |NULL       |209.9754 |1.09929000           |
|inv_n5n5ehq4|USD     |2048.9500   |25.9300    |430.2800 |1.00000000           |
|inv_03ivwack|ARS     |0.5535      |NULL       |0.1162   |0.00158000           |
|inv_ln9tlyo6|ARS     |2.4007      |NULL       |0.5041   |0.00152000           |
|inv_74dphu2x|USD     |580.0800  

In [ ]:
# Calcular métricas financieras

billing_financial = (
    billing_prepared

    # Total facturado solo si los créditos están informados
    .withColumn(
        "total_billed_usd",
        F.when(
            F.col("credits").isNotNull(),
            F.round(
                (
                    F.col("subtotal")
                    - F.col("credits")
                    + F.col("taxes")
                ) * F.col("applied_exchange_rate"),
                4
            )
        )
    )

    # Indicar si podemos calcular el total completo
    .withColumn(
        "billing_total_available",
        F.col("total_billed_usd").isNotNull()
    )
)

print("Total de facturas:", billing_financial.count())

print(
    "Facturas con total calculable:",
    billing_financial.filter(
        F.col("billing_total_available")
    ).count()
)

print(
    "Facturas sin total calculable:",
    billing_financial.filter(
        ~F.col("billing_total_available")
    ).count()
)

print("\nResumen por mes:")

billing_financial.groupBy("month").agg(
    F.count("*").alias("facturas"),

    F.round(
        F.sum("subtotal_usd"), 2
    ).alias("subtotal_usd"),

    F.count("total_billed_usd").alias(
        "totales_calculables"
    ),

    F.round(
        F.sum("total_billed_usd"), 2
    ).alias("total_conocido_usd")
).orderBy("month").show(truncate=False)


Total de facturas: 240
Facturas con total calculable: 103
Facturas sin total calculable: 137

Resumen por mes:
+----------+--------+------------+-------------------+------------------+
|month     |facturas|subtotal_usd|totales_calculables|total_conocido_usd|
+----------+--------+------------+-------------------+------------------+
|2025-06-01|80      |49142.53    |34                 |23929.56          |
|2025-07-01|80      |44772.96    |33                 |23422.86          |
|2025-08-01|80      |39802.83    |36                 |19513.87          |
+----------+--------+------------+-------------------+------------------+



## Análisis de eventos de uso

Comenzamos leyendo los 43.200 eventos de uso disponibles en Silver, correspondientes al período comprendido entre el 3 de julio y el 31 de agosto de 2025.

Validamos que todos los eventos tuvieran fecha y costo informado. También identificamos 216 eventos con costos negativos, que decidimos conservar porque podrían representar ajustes o correcciones.

A partir de estos datos construimos el primer mart Gold, llamado `org_daily_usage_by_service`, agrupando los eventos por organización, fecha y servicio.

Obtuvimos 11.050 registros agregados, que representan los 43.200 eventos originales y un costo total de USD 147.433,9778.

Verificamos que no se perdieran eventos ni costos durante la transformación.

In [ ]:
# Construir mart mensual de facturación

gold_monthly_billing = (
    billing_financial
    .groupBy("org_id", "month")
    .agg(
        F.count("*").alias("invoice_count"),

        F.round(
            F.sum("subtotal_usd"), 4
        ).alias("subtotal_usd"),

        F.round(
            F.sum("taxes_usd"), 4
        ).alias("taxes_usd"),

        F.round(
            F.sum("credits_usd"), 4
        ).alias("known_credits_usd"),

        F.round(
            F.sum("total_billed_usd"), 4
        ).alias("known_total_billed_usd"),

        F.sum(
            F.when(
                F.col("credits_missing_flag"), 1
            ).otherwise(0)
        ).alias("missing_credits_count"),

        F.sum(
            F.when(
                F.col("usd_exchange_rate_flag"), 1
            ).otherwise(0)
        ).alias("usd_exchange_rate_flags"),

        F.sum(
            F.when(
                F.col("requires_review"), 1
            ).otherwise(0)
        ).alias("invoices_requires_review")
    )
)

print("Registros del mart:", gold_monthly_billing.count())

print(
    "Facturas representadas:",
    gold_monthly_billing.agg(
        F.sum("invoice_count")
    ).first()[0]
)

print(
    "Créditos faltantes:",
    gold_monthly_billing.agg(
        F.sum("missing_credits_count")
    ).first()[0]
)

print(
    "Facturas en revisión:",
    gold_monthly_billing.agg(
        F.sum("invoices_requires_review")
    ).first()[0]
)

gold_monthly_billing.orderBy(
    "org_id", "month"
).show(10, truncate=False)


Registros del mart: 240
Facturas representadas: 240
Créditos faltantes: 137
Facturas en revisión: 13
+------------+----------+-------------+------------+---------+-----------------+----------------------+---------------------+-----------------------+------------------------+
|org_id      |month     |invoice_count|subtotal_usd|taxes_usd|known_credits_usd|known_total_billed_usd|missing_credits_count|usd_exchange_rate_flags|invoices_requires_review|
+------------+----------+-------------+------------+---------+-----------------+----------------------+---------------------+-----------------------+------------------------+
|org_0lvsnujz|2025-06-01|1            |2.0004      |0.4201   |NULL             |NULL                  |1                    |0                      |0                       |
|org_0lvsnujz|2025-07-01|1            |1600.2200   |336.0500 |21.0000          |1915.2700             |0                    |1                      |0                       |
|org_0lvsnujz|2025-08-01

In [ ]:
# Definir ruta del mart mensual de facturación

ruta_gold_billing = GOLD_PATH / "org_monthly_billing"

# Guardar los resultados en Parquet

(
    gold_monthly_billing
    .write
    .mode("overwrite")
    .parquet(str(ruta_gold_billing))
)

print("Mart de facturación guardado en:")
print(ruta_gold_billing)


Mart de facturación guardado en:
/content/drive/MyDrive/cloud-provider-analytics-data/gold/org_monthly_billing


In [ ]:
# Leer nuevamente el mart desde Google Drive

gold_billing_check = spark.read.parquet(
    str(ruta_gold_billing)
)

print("Registros guardados:", gold_billing_check.count())

print(
    "Facturas representadas:",
    gold_billing_check.agg(
        F.sum("invoice_count")
    ).first()[0]
)

print(
    "Créditos faltantes:",
    gold_billing_check.agg(
        F.sum("missing_credits_count")
    ).first()[0]
)

print(
    "Facturas en revisión:",
    gold_billing_check.agg(
        F.sum("invoices_requires_review")
    ).first()[0]
)

print(
    "Tipos de cambio USD marcados:",
    gold_billing_check.agg(
        F.sum("usd_exchange_rate_flags")
    ).first()[0]
)

print("\nRegistros por mes:")

gold_billing_check.groupBy("month").count().orderBy(
    "month"
).show(truncate=False)


Registros guardados: 240
Facturas representadas: 240
Créditos faltantes: 137
Facturas en revisión: 13
Tipos de cambio USD marcados: 160

Registros por mes:
+----------+-----+
|month     |count|
+----------+-----+
|2025-06-01|80   |
|2025-07-01|80   |
|2025-08-01|80   |
+----------+-----+



In [ ]:
# Preparar costos mensuales de uso

gold_monthly_usage = (
    gold_finops_check
    .withColumn(
        "month",
        F.trunc("event_date", "month")
    )
    .groupBy("org_id", "month")
    .agg(
        F.round(
            F.sum("total_cost_usd"), 4
        ).alias("usage_cost_usd"),

        F.sum("total_events").alias(
            "usage_events"
        ),

        F.sum("negative_cost_events").alias(
            "negative_cost_events"
        ),

        F.sum("incomplete_usage_events").alias(
            "incomplete_usage_events"
        )
    )
)

print(
    "Registros mensuales de uso:",
    gold_monthly_usage.count()
)

print("\nResumen por mes:")

gold_monthly_usage.groupBy("month").agg(
    F.count("*").alias("organizaciones"),

    F.sum("usage_events").alias(
        "total_eventos"
    ),

    F.round(
        F.sum("usage_cost_usd"), 4
    ).alias("costo_total_usd")
).orderBy("month").show(truncate=False)


Registros mensuales de uso: 160

Resumen por mes:
+----------+--------------+-------------+---------------+
|month     |organizaciones|total_eventos|costo_total_usd|
+----------+--------------+-------------+---------------+
|2025-07-01|80            |20880        |71357.6385     |
|2025-08-01|80            |22320        |76076.3393     |
+----------+--------------+-------------+---------------+



In [ ]:
# Relacionar costos de uso y facturación mensual

gold_finops_comparison = (
    gold_monthly_usage.alias("usage")
    .join(
        gold_billing_check.alias("billing"),
        on=["org_id", "month"],
        how="left"
    )
    .withColumn(
        "subtotal_minus_usage_usd",
        F.round(
            F.col("subtotal_usd") - F.col("usage_cost_usd"),
            4
        )
    )
    .withColumn(
        "billing_data_missing",
        F.col("invoice_count").isNull()
    )
)

print(
    "Registros de la comparación:",
    gold_finops_comparison.count()
)

print(
    "Organizaciones sin facturación:",
    gold_finops_comparison.filter(
        F.col("billing_data_missing")
    ).count()
)

print(
    "Eventos representados:",
    gold_finops_comparison.agg(
        F.sum("usage_events")
    ).first()[0]
)

print("\nResumen mensual:")

gold_finops_comparison.groupBy("month").agg(
    F.count("*").alias("organizaciones"),

    F.round(
        F.sum("usage_cost_usd"), 2
    ).alias("costo_uso_usd"),

    F.round(
        F.sum("subtotal_usd"), 2
    ).alias("subtotal_facturado_usd"),

    F.round(
        F.sum("subtotal_minus_usage_usd"), 2
    ).alias("diferencia_usd")
).orderBy("month").show(truncate=False)


Registros de la comparación: 160
Organizaciones sin facturación: 0
Eventos representados: 43200

Resumen mensual:
+----------+--------------+-------------+----------------------+--------------+
|month     |organizaciones|costo_uso_usd|subtotal_facturado_usd|diferencia_usd|
+----------+--------------+-------------+----------------------+--------------+
|2025-07-01|80            |71357.64     |44772.96              |-26584.68     |
|2025-08-01|80            |76076.34     |39802.83              |-36273.51     |
+----------+--------------+-------------+----------------------+--------------+



In [ ]:
# Analizar las mayores diferencias por organización y mes

print("Organizaciones con mayores diferencias negativas:")

gold_finops_comparison.select(
    "org_id",
    "month",
    "usage_cost_usd",
    "subtotal_usd",
    "subtotal_minus_usage_usd",
    "missing_credits_count",
    "usd_exchange_rate_flags",
    "invoices_requires_review"
).orderBy(
    F.col("subtotal_minus_usage_usd").asc()
).show(10, truncate=False)


print("\nOrganizaciones con mayores diferencias positivas:")

gold_finops_comparison.select(
    "org_id",
    "month",
    "usage_cost_usd",
    "subtotal_usd",
    "subtotal_minus_usage_usd",
    "missing_credits_count",
    "usd_exchange_rate_flags",
    "invoices_requires_review"
).orderBy(
    F.col("subtotal_minus_usage_usd").desc()
).show(10, truncate=False)


Organizaciones con mayores diferencias negativas:
+------------+----------+--------------+------------+------------------------+---------------------+-----------------------+------------------------+
|org_id      |month     |usage_cost_usd|subtotal_usd|subtotal_minus_usage_usd|missing_credits_count|usd_exchange_rate_flags|invoices_requires_review|
+------------+----------+--------------+------------+------------------------+---------------------+-----------------------+------------------------+
|org_chj755nf|2025-08-01|2620.1929     |2.3458      |-2617.8471              |1                    |0                      |0                       |
|org_ly8ozcyw|2025-08-01|1437.5345     |-1093.0193  |-2530.5538              |1                    |0                      |1                       |
|org_x7eedtjv|2025-08-01|1153.8217     |-879.9700   |-2033.7917              |0                    |1                      |1                       |
|org_53lc58dr|2025-08-01|2180.2765     |217.5800  

In [ ]:
# Incorporar indicadores de calidad y diferencias

gold_finops_final = (
    gold_finops_comparison

    .withColumn(
        "billing_quality_flag",
        (F.col("missing_credits_count") > 0)
        | (F.col("usd_exchange_rate_flags") > 0)
        | (F.col("invoices_requires_review") > 0)
        | F.col("billing_data_missing")
    )

    .withColumn(
        "difference_category",
        F.when(
            F.col("subtotal_minus_usage_usd") < -0.01,
            "Negative difference"
        )
        .when(
            F.col("subtotal_minus_usage_usd") > 0.01,
            "Positive difference"
        )
        .otherwise("Near zero")
    )
)

print("Registros:", gold_finops_final.count())

print("\nClasificación de diferencias:")
gold_finops_final.groupBy(
    "difference_category"
).count().show()

print("\nIndicadores de calidad:")
gold_finops_final.groupBy(
    "billing_quality_flag"
).count().show()

print("\nDiferencias por categoría y calidad:")
gold_finops_final.groupBy(
    "difference_category",
    "billing_quality_flag"
).count().orderBy(
    "difference_category",
    "billing_quality_flag"
).show()


Registros: 160

Clasificación de diferencias:
+-------------------+-----+
|difference_category|count|
+-------------------+-----+
|Positive difference|   49|
|Negative difference|  111|
+-------------------+-----+


Indicadores de calidad:
+--------------------+-----+
|billing_quality_flag|count|
+--------------------+-----+
|                true|  138|
|               false|   22|
+--------------------+-----+


Diferencias por categoría y calidad:
+-------------------+--------------------+-----+
|difference_category|billing_quality_flag|count|
+-------------------+--------------------+-----+
|Negative difference|               false|   15|
|Negative difference|                true|   96|
|Positive difference|               false|    7|
|Positive difference|                true|   42|
+-------------------+--------------------+-----+



In [ ]:
# Definir ruta del cuarto mart Gold

ruta_gold_comparison = (
    GOLD_PATH / "org_monthly_finops_comparison"
)

# Guardar la comparación mensual en Parquet

(
    gold_finops_final
    .write
    .mode("overwrite")
    .parquet(str(ruta_gold_comparison))
)

print("Mart de comparación guardado en:")
print(ruta_gold_comparison)


Mart de comparación guardado en:
/content/drive/MyDrive/cloud-provider-analytics-data/gold/org_monthly_finops_comparison


In [ ]:
# Leer nuevamente el mart desde Google Drive

gold_comparison_check = spark.read.parquet(
    str(ruta_gold_comparison)
)

print("Registros guardados:", gold_comparison_check.count())

print(
    "Eventos representados:",
    gold_comparison_check.agg(
        F.sum("usage_events")
    ).first()[0]
)

print(
    "Organizaciones sin facturación:",
    gold_comparison_check.filter(
        F.col("billing_data_missing")
    ).count()
)

print("\nResumen mensual:")

gold_comparison_check.groupBy("month").agg(
    F.count("*").alias("organizaciones"),

    F.round(
        F.sum("usage_cost_usd"), 2
    ).alias("costo_uso_usd"),

    F.round(
        F.sum("subtotal_usd"), 2
    ).alias("subtotal_facturado_usd"),

    F.round(
        F.sum("subtotal_minus_usage_usd"), 2
    ).alias("diferencia_usd")
).orderBy("month").show(truncate=False)


Registros guardados: 160
Eventos representados: 43200
Organizaciones sin facturación: 0

Resumen mensual:
+----------+--------------+-------------+----------------------+--------------+
|month     |organizaciones|costo_uso_usd|subtotal_facturado_usd|diferencia_usd|
+----------+--------------+-------------+----------------------+--------------+
|2025-07-01|80            |71357.64     |44772.96              |-26584.68     |
|2025-08-01|80            |76076.34     |39802.83              |-36273.51     |
+----------+--------------+-------------+----------------------+--------------+



In [ ]:
# Validación final del mart de comparación FinOps

print("=== VALIDACIÓN FINAL GOLD FINOPS ===")

# 1. Total de registros
total_registros = gold_comparison_check.count()
print("Registros totales:", total_registros)

# 2. Duplicados por organización y mes
duplicados = (
    gold_comparison_check
    .groupBy("org_id", "month")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Duplicados org_id + month:", duplicados)

# 3. Valores nulos en columnas clave
columnas_clave = [
    "org_id",
    "month",
    "usage_cost_usd",
    "subtotal_usd",
    "subtotal_minus_usage_usd"
]

for columna in columnas_clave:
    nulos = gold_comparison_check.filter(
        F.col(columna).isNull()
    ).count()

    print(f"Nulos en {columna}: {nulos}")

# 4. Validar períodos
print("\nPeríodos disponibles:")

gold_comparison_check.groupBy("month").count().orderBy(
    "month"
).show()

# 5. Verificar costo y eventos
gold_comparison_check.agg(
    F.sum("usage_events").alias("total_eventos"),
    F.round(
        F.sum("usage_cost_usd"), 4
    ).alias("total_costo_usd")
).show()


=== VALIDACIÓN FINAL GOLD FINOPS ===
Registros totales: 160
Duplicados org_id + month: 0
Nulos en org_id: 0
Nulos en month: 0
Nulos en usage_cost_usd: 0
Nulos en subtotal_usd: 0
Nulos en subtotal_minus_usage_usd: 0

Períodos disponibles:
+----------+-----+
|     month|count|
+----------+-----+
|2025-07-01|   80|
|2025-08-01|   80|
+----------+-----+

+-------------+---------------+
|total_eventos|total_costo_usd|
+-------------+---------------+
|        43200|    147433.9778|
+-------------+---------------+




## Conclusiones y validaciones de la capa Gold

En este notebook construimos cuatro marts Gold para analizar los costos de uso y la facturación de las organizaciones de Cloud Provider Analytics.

Procesamos 43.200 eventos de uso y obtuvimos un costo total de USD 147.433,9778, distribuido entre julio y agosto de 2025.

También procesamos 240 facturas correspondientes a junio, julio y agosto de 2025, identificando créditos faltantes, subtotales negativos y posibles inconsistencias en los tipos de cambio.

Posteriormente, relacionamos los costos de uso con la facturación mensual de las 80 organizaciones, obteniendo 160 registros de comparación.

De estos registros, 111 presentaron diferencias negativas y 49 diferencias positivas. Además, 138 tenían al menos una alerta de calidad de facturación.

Estas diferencias no deben interpretarse directamente como ganancias o pérdidas, ya que los conceptos comparados pueden ser diferentes y existen reglas de negocio pendientes de confirmar.

### Validaciones finales

- 160 registros de comparación.
- 0 duplicados por organización y mes.
- 0 valores nulos en las columnas clave.
- 80 organizaciones por mes.
- 43.200 eventos conservados.
- USD 147.433,9778 de costos conservados.

Las validaciones estructurales fueron satisfactorias.

Como próximos pasos, se propone revisar las decisiones de calidad pendientes, documentar las reglas de transformación y preparar la integración de los datos Gold con Cassandra/AstraDB.
